In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier
import optuna

In [ ]:
dataset = pd.read_csv('/content/train.csv')

In [ ]:
def make_churn_to_binary(x):
  if x == 'Yes':
    return 1
  else:
    return 0

In [ ]:
dataset

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,594189,Male,0,No,No,57,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Two year,No,Bank transfer (automatic),97.55,5460.70,No
594190,594190,Female,0,No,No,72,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,No,Bank transfer (automatic),91.95,6782.15,No
594191,594191,Female,0,Yes,No,72,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),24.40,1871.90,No
594192,594192,Female,0,No,No,32,Yes,Yes,Fiber optic,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,86.00,2847.20,No


In [ ]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

In [ ]:
dataset.drop('id', axis = 1, inplace = True)

In [ ]:
for col in dataset.columns:
  if dataset[col].dtype == 'object':
    print(col, dataset[col].unique(),' - ', dataset[col].unique().shape[0])

gender ['Male' 'Female']  -  2
Partner ['Yes' 'No']  -  2
Dependents ['Yes' 'No']  -  2
PhoneService ['Yes' 'No']  -  2
MultipleLines ['No' 'Yes' 'No phone service']  -  3
InternetService ['DSL' 'Fiber optic' 'No']  -  3
OnlineSecurity ['Yes' 'No' 'No internet service']  -  3
OnlineBackup ['No' 'Yes' 'No internet service']  -  3
DeviceProtection ['Yes' 'No' 'No internet service']  -  3
TechSupport ['Yes' 'No' 'No internet service']  -  3
StreamingTV ['No' 'Yes' 'No internet service']  -  3
StreamingMovies ['No' 'Yes' 'No internet service']  -  3
Contract ['One year' 'Two year' 'Month-to-month']  -  3
PaperlessBilling ['Yes' 'No']  -  2
PaymentMethod ['Mailed check' 'Credit card (automatic)' 'Electronic check'
 'Bank transfer (automatic)']  -  4
Churn ['No' 'Yes']  -  2


In [ ]:
x = dataset.drop('Churn', axis = 1)
y = dataset['Churn']

In [ ]:
y = y.apply(make_churn_to_binary)

In [ ]:
one_hot_cols = []
for col in x.columns:
  if x[col].dtype == 'object':
    one_hot_cols.append(col);
one_hot_cols

['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

In [ ]:
ct = ColumnTransformer([
    ('encoder', OneHotEncoder(drop='if_binary'), one_hot_cols)
], remainder='passthrough')

In [ ]:
x = ct.fit_transform(x)
raw_names = ct.get_feature_names_out()


In [ ]:
raw_names

array(['encoder__gender_Male', 'encoder__Partner_Yes',
       'encoder__Dependents_Yes', 'encoder__PhoneService_Yes',
       'encoder__MultipleLines_No',
       'encoder__MultipleLines_No phone service',
       'encoder__MultipleLines_Yes', 'encoder__InternetService_DSL',
       'encoder__InternetService_Fiber optic',
       'encoder__InternetService_No', 'encoder__OnlineSecurity_No',
       'encoder__OnlineSecurity_No internet service',
       'encoder__OnlineSecurity_Yes', 'encoder__OnlineBackup_No',
       'encoder__OnlineBackup_No internet service',
       'encoder__OnlineBackup_Yes', 'encoder__DeviceProtection_No',
       'encoder__DeviceProtection_No internet service',
       'encoder__DeviceProtection_Yes', 'encoder__TechSupport_No',
       'encoder__TechSupport_No internet service',
       'encoder__TechSupport_Yes', 'encoder__StreamingTV_No',
       'encoder__StreamingTV_No internet service',
       'encoder__StreamingTV_Yes', 'encoder__StreamingMovies_No',
       'encoder__St

In [ ]:
cols = [name.split('__')[-1] for name in raw_names]

In [ ]:
cols

['gender_Male',
 'Partner_Yes',
 'Dependents_Yes',
 'PhoneService_Yes',
 'MultipleLines_No',
 'MultipleLines_No phone service',
 'MultipleLines_Yes',
 'InternetService_DSL',
 'InternetService_Fiber optic',
 'InternetService_No',
 'OnlineSecurity_No',
 'OnlineSecurity_No internet service',
 'OnlineSecurity_Yes',
 'OnlineBackup_No',
 'OnlineBackup_No internet service',
 'OnlineBackup_Yes',
 'DeviceProtection_No',
 'DeviceProtection_No internet service',
 'DeviceProtection_Yes',
 'TechSupport_No',
 'TechSupport_No internet service',
 'TechSupport_Yes',
 'StreamingTV_No',
 'StreamingTV_No internet service',
 'StreamingTV_Yes',
 'StreamingMovies_No',
 'StreamingMovies_No internet service',
 'StreamingMovies_Yes',
 'Contract_Month-to-month',
 'Contract_One year',
 'Contract_Two year',
 'PaperlessBilling_Yes',
 'PaymentMethod_Bank transfer (automatic)',
 'PaymentMethod_Credit card (automatic)',
 'PaymentMethod_Electronic check',
 'PaymentMethod_Mailed check',
 'SeniorCitizen',
 'tenure',
 'Mo

In [ ]:
new_df = pd.DataFrame(x, columns = cols, index = dataset.index)

In [ ]:
new_df

,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,InternetService_Fiber optic,InternetService_No,...,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,29.0,60.10,1653.85
1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,58.0,69.50,3778.20
2,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,58.0,100.40,5841.35
3,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,69.70,70.70
4,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,70.45,70.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,57.0,97.55,5460.70
594190,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,72.0,91.95,6782.15
594191,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,72.0,24.40,1871.90
594192,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,32.0,86.00,2847.20


In [ ]:
for col in new_df.columns:
  if new_df[col].dtype == 'float64':
    new_df[col] = new_df[col].astype(int)

In [ ]:
new_df

,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,InternetService_DSL,InternetService_Fiber optic,InternetService_No,...,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
0,1,1,1,1,1,0,0,1,0,0,...,0,1,0,0,0,1,0,29,60,1653
1,1,1,1,1,1,0,0,1,0,0,...,1,0,0,1,0,0,0,58,69,3778
2,1,1,0,1,0,0,1,0,1,0,...,0,1,0,0,1,0,0,58,100,5841
3,0,0,0,1,1,0,0,0,1,0,...,0,1,0,0,1,0,0,1,69,70
4,0,0,0,1,1,0,0,0,1,0,...,0,1,0,0,1,0,0,1,70,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,1,0,0,1,0,0,1,0,1,0,...,1,0,1,0,0,0,0,57,97,5460
594190,0,0,0,1,0,0,1,1,0,0,...,1,0,1,0,0,0,0,72,91,6782
594191,0,1,0,1,0,0,1,0,0,1,...,1,0,0,1,0,0,0,72,24,1871
594192,0,0,0,1,0,0,1,0,1,0,...,0,1,0,0,1,0,0,32,86,2847


In [ ]:
models = [
    ('randomforest', RandomForestClassifier(n_estimators=50)),
    ('xgboost', XGBClassifier()),
    ('gradientboost', GradientBoostingClassifier(n_estimators=50, learning_rate=0.08)),
]

In [ ]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, stratify=y, test_size=0.2, random_state = 43)

In [ ]:
model = XGBClassifier()
model.fit(xtrain, ytrain)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# Use predict_proba instead of predict
y_probs = model.predict_proba(xtest)[:, 1]

# Calculate ROC AUC using probabilities
roc_auc = roc_auc_score(ytest, y_probs)
print(f"ROC AUC Score: {roc_auc:.4f}")

ROC AUC Score: 0.9165


In [ ]:
final_model = XGBClassifier()
final_model.fit(x, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
test = pd.read_csv('/content/test.csv')

In [ ]:
id = test.id

In [ ]:
test.drop('id', axis = 1, inplace = True)

In [ ]:
text = ct.transform(test)
raw_names_test = ct.get_feature_names_out()

In [ ]:
col_names = [name.split('__')[-1] for name in raw_names_test]

In [ ]:
test_df = pd.DataFrame(text, columns = col_names, index = test.index)

In [ ]:
final_preds = final_model.predict_proba(test_df)[:, 1]

In [ ]:
final_preds


array([6.69981241e-02, 3.46588495e-04, 1.07777484e-01, ...,
       2.48405740e-01, 1.41998578e-03, 3.51089895e-01], dtype=float32)

In [ ]:
submission = pd.DataFrame({
    'id': id,
    'Churn': final_preds
})
submission.to_csv('submission.csv', index=False)